# 08 · 端到端實戰:完整影像分類專案

整條軌道的收尾。把學到的東西串成一個**完整流程**:遷移學習 + 資料增強訓練一個分類器,對單張圖做推論,存檔,最後聊怎麼**部署到瀏覽器**——呼應本站的做法。

## 1. 安裝與資料(遷移學習 + 增強)

In [ ]:
%pip install -q -U torchvision

In [ ]:
# CIFAR-10 的 10 個類別，以及這個資料集慣用的正規化平均/標準差。
CIFAR10_CLASSES = ["plane", "car", "bird", "cat", "deer",
                   "dog", "frog", "horse", "ship", "truck"]
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

In [ ]:
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.models import resnet18, ResNet18_Weights

device = "cuda" if torch.cuda.is_available() else "cpu"
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

train_tf = transforms.Compose([
    transforms.Resize(224), transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
test_tf = transforms.Compose([
    transforms.Resize(224), transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
train_set = torchvision.datasets.CIFAR10("./data", train=True, download=True, transform=train_tf)
test_set = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=test_tf)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_set, batch_size=256, num_workers=2)

## 2. 組裝模型:預訓練 ResNet-18 + 新頭

In [ ]:
model = resnet18(weights=ResNet18_Weights.DEFAULT)
for p in model.parameters():
    p.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

opt = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
for ep in range(1, 4):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad(); loss_fn(model(x), y).backward(); opt.step()
    print(f"epoch {ep}/3 完成")
print("訓練完成。")

## 3. 評估

In [ ]:
model.eval()
correct = total = 0
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
        total += y.size(0)
print(f"測試準確率:{100 * correct / total:.1f}%")

## 4. 單張圖推論 demo:Top-3 預測

拿一張測試圖,看模型最有把握的前三名。

In [ ]:
import matplotlib.pyplot as plt

raw = torchvision.datasets.CIFAR10("./data", train=False)
pil_img, true_label = raw[12]
x = test_tf(pil_img).unsqueeze(0).to(device)
with torch.no_grad():
    probs = model(x).softmax(1)[0]
top3 = probs.topk(3)
plt.imshow(pil_img); plt.axis("off")
plt.title("真實: " + CIFAR10_CLASSES[true_label]); plt.show()
print("Top-3 預測:")
for p, idx in zip(top3.values, top3.indices):
    print(f"  {CIFAR10_CLASSES[int(idx)]:8s} {float(p):.2%}")

## 5. 存檔 + 部署思路

In [ ]:
torch.save(model.state_dict(), "cifar_resnet18.pt")
print("模型已存檔。")

# 要搬上網頁?把模型匯出成 ONNX,前端用 onnxruntime-web 或轉 TF.js 即可瀏覽器即時推論:
# dummy = torch.randn(1, 3, 224, 224, device=device)
# torch.onnx.export(model, dummy, "cifar_resnet18.onnx", input_names=["image"])

## 從 Colab 到瀏覽器:本站怎麼做

本站的 **MNIST 手寫辨識**等小工具,就是把訓練好的視覺模型搬進瀏覽器即時跑:

- 模型在 Python/Colab 訓練好後,**匯出成 ONNX 或 TF.js 格式**。
- 前端用 **TensorFlow.js / onnxruntime-web** 載入,在使用者瀏覽器**本地推論**——不需要伺服器、不上傳影像、零延遲。
- 這跟 `rl` 軌道把 agent 權重匯出到瀏覽器是同一套思路:**Colab 訓練 → 匯出 → 前端即時推論**。

## 軌道小結

你從**影像即張量**,一路做到能解釋、能偵測、能上線:

- **影像張量 / 前處理**(01)→ **CNN on CIFAR**(02)
- **遷移學習**(03)→ **資料增強**(04):用更少資源做更準的分類
- **物件偵測 YOLO**(05)→ **影像分割**(06):從「是什麼」到「在哪裡、什麼形狀」
- **Grad-CAM 可解釋性**(07)→ **端到端專案 + 部署**(08)

**會用預訓練模型、懂遷移學習、能解釋與部署**——這正是業界電腦視覺工程師的日常工具箱。📷